# Inspect trained BEV-Gemma QA model

Loads checkpoints from `vla.bev_qa.train` and prints question / ground-truth / prediction on validation samples.

Edit `OUTPUT_DIR`, `QA_DIR`, and `IMG_ROOT` below, then run cells in order.

In [1]:
!hostname
!nvidia-smi

rhea.int.autonlab.org


/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import json
import os
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "vla").is_dir():
            return candidate
    raise RuntimeError(
        f"Could not find repo root (no 'vla/' package) starting from {start.resolve()}"
    )


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT={REPO_ROOT}")

from vla.bev_qa.dataloader import BEVQADataset, bev_qa_collate_fn
from vla.bev_qa.model import BEVGemmaQA
from vla.bev_qa.train import _resolve_dtype, _split_indices

OUTPUT_DIR = Path("/zfsauton/scratch/sbellad/waymax_rs/bev_qa_output_gemma")
QA_DIR = "/zfsauton/scratch/mineuih/waymax_rs/qa_dataset/"
IMG_ROOT = "/zfsauton/scratch/eshau/imgs_past/"
FILE_INDICES = None
VALIDATION_FRACTION = 0.04
SEED = 0

NUM_EXAMPLES = 100
MAX_PROMPT_LENGTH = 128
MAX_ANSWER_LENGTH = 8
BATCH_SIZE = 1

# None = latest checkpoint; or set e.g. 27055 for step_00027055.pt
CHECKPOINT_STEP = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = _resolve_dtype("bf16") if device.type == "cuda" else torch.float32
print(f"device={device}, dtype={dtype}")


REPO_ROOT=/zfsauton2/home/sbellad/waymax_rs-main


device=cuda, dtype=torch.bfloat16


In [3]:


def load_bev_training_config(output_dir: Path, checkpoint: dict) -> dict:
    config_path = output_dir / "config.json"
    if config_path.exists():
        with open(config_path, encoding="utf-8") as f:
            return json.load(f)
    if "config" in checkpoint:
        return checkpoint["config"]
    raise FileNotFoundError(f"No config.json in {output_dir} and checkpoint has no config.")


def find_checkpoint(output_dir: Path, step: int | None = None) -> Path:
    ckpt_dir = output_dir / "checkpoints"
    if step is not None:
        path = ckpt_dir / f"step_{int(step):08d}.pt"
        if not path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {path}")
        return path

    checkpoints = sorted(ckpt_dir.glob("step_*.pt"))
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")
    return checkpoints[-1]


checkpoint_path = find_checkpoint(OUTPUT_DIR, CHECKPOINT_STEP)
print(f"Loading checkpoint: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
train_config = load_bev_training_config(OUTPUT_DIR, checkpoint)

GEMMA_NAME = train_config.get("gemma_name", "google/gemma-4-E2B-it")
FREEZE_GEMMA = train_config.get("freeze_gemma", True)
FREEZE_VISION = train_config.get("freeze_vision", True)
MAX_PROMPT_LENGTH = train_config.get("max_prompt_length", MAX_PROMPT_LENGTH)
MAX_ANSWER_LENGTH = train_config.get("max_answer_length", MAX_ANSWER_LENGTH)
VALIDATION_FRACTION = train_config.get("validation_fraction", VALIDATION_FRACTION)
SEED = train_config.get("seed", SEED)
QA_DIR = train_config.get("qa_dir", QA_DIR)
IMG_ROOT = train_config.get("img_root", IMG_ROOT)
FILE_INDICES = train_config.get("file_indices", FILE_INDICES)

model = BEVGemmaQA(
    gemma_name=GEMMA_NAME,
    freeze_gemma=FREEZE_GEMMA,
    freeze_vision=FREEZE_VISION,
)

if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

state_key = "trainable_model_state_dict"
if state_key not in checkpoint:
    raise KeyError(
        f"Checkpoint missing {state_key!r}. Keys: {list(checkpoint.keys())}"
    )

missing, unexpected = model.load_state_dict(
    checkpoint[state_key],
    strict=False,
)
if missing:
    print(f"Warning: {len(missing)} missing keys (expected for frozen Gemma weights)")
if unexpected:
    raise RuntimeError(f"Unexpected keys in checkpoint: {unexpected}")

model = model.to(device=device, dtype=dtype)
model.eval()

print("Model loaded successfully.")
print(f"Checkpoint step: {checkpoint.get('step', 'unknown')}")
print(f"gemma_name={GEMMA_NAME}")
print(f"trainable tensors loaded: {len(checkpoint[state_key])}")


Loading checkpoint: /zfsauton/scratch/sbellad/waymax_rs/bev_qa_output_gemma/checkpoints/step_00027055.pt


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[BEVGemmaQA] hidden_size=1536
[BEVGemmaQA] vision_hidden_size=768
[BEVGemmaQA] vision_soft_tokens_per_image=280


Model loaded successfully.
Checkpoint step: 27055
gemma_name=google/gemma-4-E2B-it
trainable tensors loaded: 12


In [4]:
dataset = BEVQADataset(
    qa_dir=QA_DIR,
    img_root=IMG_ROOT,
    file_indices=FILE_INDICES,
    one_qa_per_scenario=True,
    seed=SEED,
)

_, val_indices = _split_indices(len(dataset), VALIDATION_FRACTION, SEED)
inspection_indices = val_indices if val_indices else list(range(len(dataset)))
inspection_ds = Subset(dataset, inspection_indices)

loader = DataLoader(
    inspection_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=bev_qa_collate_fn,
)

print(f"dataset samples={len(dataset)}")
print(f"inspection samples={len(inspection_ds)} (validation_fraction={VALIDATION_FRACTION})")
print(f"missing_images={dataset.missing_images}, skipped_no_qas={dataset.skipped_no_qas}")

shown = 0
correct = 0

def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().strip().split())


with torch.inference_mode():
    for batch in loader:
        remaining = NUM_EXAMPLES - shown
        if remaining <= 0:
            break

        images = batch["images"][:remaining]
        questions = batch["questions"][:remaining]
        answers = batch["answers"][:remaining]
        qa_keys = batch["qa_keys"][:remaining]

        amp_enabled = device.type == "cuda"
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=amp_enabled):
            predictions = model.generate_answers(
                images=images,
                questions=questions,
                max_prompt_length=MAX_PROMPT_LENGTH,
                max_new_tokens=MAX_ANSWER_LENGTH,
            )

        for question, answer, prediction, qa_key in zip(
            questions, answers, predictions, qa_keys
        ):
            match = _normalize_text(prediction) == _normalize_text(answer)
            correct += int(match)

            print(f"[key={qa_key}]")
            print(f"Q:   {question}")
            print(f"GT:  {answer}")
            print(f"Pred:{prediction}")
            print("-" * 88)
            shown += 1

        if shown >= NUM_EXAMPLES:
            break

accuracy = correct / shown if shown > 0 else 0.0
print(f"\nExact-match accuracy on {shown} examples: {accuracy:.3f} ({correct}/{shown})")


Loading BEV-QA data:   0%|                                                                                                                                                       | 0/130 [00:00<?, ?file/s]


Loading BEV-QA data:   2%|██▍                                                                                                      | 3/130 [00:00<00:04, 26.46file/s, missing=0, samples=1312, skipped=136]


Loading BEV-QA data:   5%|████▊                                                                                                    | 6/130 [00:00<00:08, 15.27file/s, missing=0, samples=2626, skipped=261]


Loading BEV-QA data:   6%|██████▍                                                                                                  | 8/130 [00:00<00:08, 14.14file/s, missing=0, samples=3500, skipped=371]


Loading BEV-QA data:   8%|████████                                                                                                | 10/130 [00:00<00:08, 13.80file/s, missing=0, samples=4389, skipped=462]

...

[key=num_vehicle_front_same_lane]
Q:   How many vehicles are in front of the ego vehicle in the same lane?
GT:  0
Pred:0.000 0.
----------------------------------------------------------------------------------------


[key=num_vehicle_behind_same_lane]
Q:   How many vehicles are behind the ego vehicle in the same lane?
GT:  0
Pred:1.1310 -1
----------------------------------------------------------------------------------------


[key=traffic_light_state]
Q:   What is the traffic light state class? Answer with the integer class.
GT:  -1
Pred:-1.00 -1.
----------------------------------------------------------------------------------------


[key=num_vehicle_right]
Q:   How many vehicles are on the right side of the ego vehicle?
GT:  0
Pred:0.0000 0
----------------------------------------------------------------------------------------


[key=num_vehicle_right]
Q:   How many vehicles are on the right side of the ego vehicle?
GT:  0
Pred:0.0000 0
----------------------------------------------------------------------------------------

Exact-match accuracy on 100 examples: 0.000 (0/100)


In [5]:
ckpt_dir = OUTPUT_DIR / "checkpoints"
checkpoints = sorted(ckpt_dir.glob("step_*.pt"))
print(f"Found {len(checkpoints)} checkpoints in {ckpt_dir}")
for path in checkpoints:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"  {path.name}  step={ckpt.get('step', '?')}")

Found 28 checkpoints in /zfsauton/scratch/sbellad/waymax_rs/bev_qa_output_gemma/checkpoints


  step_00001000.pt  step=1000
  step_00002000.pt  step=2000
  step_00003000.pt  step=3000
  step_00004000.pt  step=4000


  step_00005000.pt  step=5000
  step_00006000.pt  step=6000
  step_00007000.pt  step=7000
  step_00008000.pt  step=8000


  step_00009000.pt  step=9000
  step_00010000.pt  step=10000
  step_00011000.pt  step=11000
  step_00012000.pt  step=12000
  step_00013000.pt  step=13000


  step_00014000.pt  step=14000
  step_00015000.pt  step=15000
  step_00016000.pt  step=16000
  step_00017000.pt  step=17000
  step_00018000.pt  step=18000


  step_00019000.pt  step=19000
  step_00020000.pt  step=20000
  step_00021000.pt  step=21000
  step_00022000.pt  step=22000
  step_00023000.pt  step=23000


  step_00024000.pt  step=24000
  step_00025000.pt  step=25000
  step_00026000.pt  step=26000
  step_00027000.pt  step=27000
  step_00027055.pt  step=27055
